# 04 — Baseline forecasts (14-day holdout)

**Series:** hourly `load_mw` (`data/raw/de_lu_load_hourly.parquet`).

**Test set:** the final **14 × 24 = 336** hours. All forecasts use only history **strictly before** each test timestamp (extended history from the train period for seasonal lags).

**Baselines**
- **Naïve:** last observed value $\hat{y}_t = y_{t-1}$
- **Seasonal naïve (lag 24):** same hour yesterday $\hat{y}_t = y_{t-24}$
- **Seasonal naïve (lag 168):** same hour last week $\hat{y}_t = y_{t-168}$
- **Drift:** $\hat{y}_{T+h} = y_T + h\,(y_T - y_1)/(T-1)$ with train length $T$ and horizon $h=1,\ldots$ into the holdout (`y_1`,`y_T` are first and last train values).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import sys

_SRC = Path.cwd().resolve() / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from utils import load_hourly_load_series, score



In [ ]:
TZ = "Europe/Berlin"
HOLDOUT_DAYS = 14 #  the number of days to hold out for testing

y = load_hourly_load_series(tz=TZ)

n_test = HOLDOUT_DAYS * 24
if len(y) <= n_test + 168:
    raise ValueError(f"Need more than {n_test + 168} hours for train + test + lag-168.")

train = y.iloc[:-n_test]
test = y.iloc[-n_test:]
y_all = y.values.astype(float) #  first extracts the numerical data from the pandas Series y as a NumPy array using .values,

print("Rows:", len(y), "| Train:", len(train), "| Test:", len(test))
print("Test window:", test.index.min(), "->", test.index.max())


In [ ]:
n = len(y_all)
base = n - n_test  # index of first test hour in full series

# Naive: lag-1
pred_naive = y_all[base - 1 : n - 1]

# Seasonal naive: lag-24 and lag-168
pred_sn24 = y_all[base - 24 : n - 24]
pred_sn168 = y_all[base - 168 : n - 168]

# Drift: Hyndman & Athanasopoulos — linear trend from first to last train observation
T_train = len(train)
y_first = float(train.iloc[0])
y_last = float(train.iloc[-1])
if T_train < 2:
    raise ValueError("Drift needs at least two train points.")
slope = (y_last - y_first) / (T_train - 1)
horizons = np.arange(1, n_test + 1, dtype=float)
pred_drift = y_last + horizons * slope

y_test = test.values.astype(float)
assert pred_naive.shape == y_test.shape == pred_sn24.shape == pred_sn168.shape == pred_drift.shape

In [ ]:
def score_block(name: str, y_pred: np.ndarray) -> pd.Series:
    metrics = score(y_test.values.astype(float), y_pred)
    return pd.Series({"method": name, **metrics})


results = pd.DataFrame(
    [
        score_block("Naive (lag-1)", pred_naive),
        score_block("Seasonal naive lag-24", pred_sn24),
        score_block("Seasonal naive lag-168", pred_sn168),
        score_block("Drift", pred_drift),
    ]
).set_index("method")

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
results
